In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/dirty_cafe_sales.csv")
print("Initial Shape:", df.shape)
duplicates_removed = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicates removed: {duplicates_removed}")
df.replace(["ERROR", "UNKNOWN"], np.nan, inplace=True)

print("\nMissing values after converting sentinel strings ('ERROR' / 'UNKNOWN'):")
print(df.isnull().sum())

Initial Shape: (10000, 8)
Duplicates removed: 0

Missing values after converting sentinel strings ('ERROR' / 'UNKNOWN'):
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


In [2]:
menu_prices = {
    "Cake": 3.0, "Coffee": 2.0, "Cookie": 1.0, "Juice": 3.0,
    "Salad": 5.0, "Sandwich": 4.0, "Smoothie": 4.0, "Tea": 1.5
}
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
missing_before = df["Price Per Unit"].isnull().sum()
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(menu_prices))

print(f"Missing 'Price Per Unit' before mapping: {missing_before}")
print(f"Missing 'Price Per Unit' after mapping: {df['Price Per Unit'].isnull().sum()}")

Missing 'Price Per Unit' before mapping: 533
Missing 'Price Per Unit' after mapping: 54


In [3]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
df["Quantity"] = df["Quantity"].fillna((df["Total Spent"] / df["Price Per Unit"]).round())
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median()).astype(int)
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Price Per Unit"].median())
df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])

print("Numeric missing values after reconstruction:")
print(df[["Quantity", "Price Per Unit", "Total Spent"]].isnull().sum())
print("\nSample fixed records (row 2 originally had Total Spent = 'ERROR'):")
print(df[["Transaction ID", "Item", "Quantity", "Price Per Unit", "Total Spent"]].iloc[[2]])

Numeric missing values after reconstruction:
Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64

Sample fixed records (row 2 originally had Total Spent = 'ERROR'):
  Transaction ID    Item  Quantity  Price Per Unit  Total Spent
2    TXN_4271903  Cookie         4             1.0          4.0


In [4]:
df["Item"] = df["Item"].fillna("Unknown")
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce").bfill().ffill()
df.to_csv("cleaned_cafe_sales.csv", index=False)

print("Null values remaining across all columns:")
print(df.isnull().sum())

print("\nFinal Data Types:")
print(df.dtypes)

Null values remaining across all columns:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Final Data Types:
Transaction ID              object
Item                        object
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object
